In [ ]:
!pip install datasets tiktoken -q

In [ ]:
import os, numpy as np, tiktoken
from datasets import load_dataset

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

TRAIN_BIN     = "/kaggle/working/train.bin"
VAL_BIN       = "/kaggle/working/val.bin"

In [ ]:
if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN) \
   and os.path.getsize(TRAIN_BIN) > 0 and os.path.getsize(VAL_BIN) > 0:
    print("Binary files exist, skipping tokenization.")
else:
    
    enc = tiktoken.get_encoding("gpt2")
    train_tokens, val_tokens = [], []

    print("Loading TinyStories (train + validation)...")

    train_dataset = load_dataset("roneneldan/TinyStories", split="train")
    val_dataset   = load_dataset("roneneldan/TinyStories", split="validation")

    print("Tokenizing train split...")
    for i, example in enumerate(train_dataset):
        train_tokens.extend(enc.encode(example["text"] + "\n"))

        if i % 1000 == 0:
            print(f"  train: {len(train_tokens)/1e6:.1f}M tokens")

    print("Tokenizing validation split...")
    for i, example in enumerate(val_dataset):
        val_tokens.extend(enc.encode(example["text"] + "\n"))

        if i % 1000 == 0:
            print(f"  val: {len(val_tokens)/1e6:.1f}M tokens")

    # save
    np.array(train_tokens, dtype=np.uint16).tofile(TRAIN_BIN)
    np.array(val_tokens,   dtype=np.uint16).tofile(VAL_BIN)

    print("saved.")

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")
print(f"train: {len(train_data):,} | val: {len(val_data):,}")
assert train_data.max() < 50257, f"OOB token in train: {train_data.max()}"
assert val_data.max()   < 50257, f"OOB token in val: {val_data.max()}"
print("tokens OK")

In [ ]:
script = r"""
import os, math, time, numpy as np
import torch, torch.nn as nn
from torch.nn import functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import tiktoken

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

dist.init_process_group(backend="nccl")
local_rank = int(os.environ["LOCAL_RANK"])
world_size = dist.get_world_size()
device     = f"cuda:{local_rank}"
is_master  = local_rank == 0
torch.cuda.set_device(device)

batch_size         = 32
accumulation_steps = 4
block_size         = 256
max_iters          = 150_000
eval_interval      = 500
eval_iters         = 30
eval_batch_size    = 16
learning_rate      = 3e-4
min_lr             = 3e-5
warmup_iters       = 2000
n_embd             = 512
n_head             = 8
n_layer            = 8
dropout            = 0.1
vocab_size         = 50257
checkpoint_dir     = "/kaggle/working/checkpoints"
TRAIN_BIN          = "/kaggle/working/train.bin"
VAL_BIN            = "/kaggle/working/val.bin"


train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

enc = tiktoken.get_encoding("gpt2")

period_ids = set(enc.encode("."))
q_ids      = set(enc.encode("?"))
e_ids      = set(enc.encode("!"))

SENTENCE_END_TOKENS = period_ids | q_ids | e_ids

def clean_start_index(data, i):
    # skip unwanted tokens at start
    while i < len(data) - 1:
        tok = int(data[i])

        if tok in SENTENCE_END_TOKENS:
            i += 1
        else:
            break

    return i


# using numpy op

def find_sentence_starts(data):
    data_np = np.array(data, dtype=np.int32)

    mask = np.isin(data_np[:-1], list(SENTENCE_END_TOKENS))
    starts = np.where(mask)[0] + 1

    print(f"Scanned {len(data_np)/1e6:.0f}M tokens (vectorized)")
    print(f"Found {len(starts)} sentence starts")

    return starts.astype(np.int64)


if is_master:
    print("Finding sentence boundaries (train)...")

sentence_starts_train = find_sentence_starts(train_data)

if is_master:
    print(f"Found {len(sentence_starts_train)} sentence starts.")

if is_master:
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"world_size: {world_size} | device: {device}")

def get_batch(split, bs=None):
    if bs is None:
        bs = batch_size

    data = train_data if split == "train" else val_data
    use_sentence = (split == "train")
    n_sentence = bs // 2 if use_sentence else 0
    n_random   = bs - n_sentence

    ix_list = []

    if use_sentence:
        sent_ix = torch.randint(0, len(sentence_starts_train), (n_sentence,))
        sent_ix = torch.from_numpy(sentence_starts_train)[sent_ix]

        sent_ix = torch.tensor([
            clean_start_index(data, int(i)) for i in sent_ix
        ])

        sent_ix = sent_ix[sent_ix < len(data) - block_size]
        ix_list.append(sent_ix)

    rand_ix = torch.randint(0, len(data) - block_size, (n_random,))
    ix_list.append(rand_ix)
    ix = torch.cat(ix_list, dim=0)
    ix = ix[torch.randperm(len(ix))]

    x = torch.stack([
        torch.from_numpy(data[i:i+block_size].astype(np.int64))
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64))
        for i in ix
    ])

    return x.to(device), y.to(device)


# model

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn     = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj     = nn.Linear(n_embd, n_embd)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(
            torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(n_embd, dim=2)
        k = k.view(B, T, n_head, C // n_head).transpose(1, 2)
        q = q.view(B, T, n_head, C // n_head).transpose(1, 2)
        v = v.view(B, T, n_head, C // n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.drop   = nn.Dropout(dropout)
        self.act    = nn.GELU()
    def forward(self, x):
        return self.drop(self.c_proj(self.act(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.mlp  = MLP()
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(
            self.token_embedding(idx) +
            self.position_embedding(torch.arange(T, device=device))
        )
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1)) \
               if targets is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
            idx = torch.cat((idx,
                torch.multinomial(F.softmax(logits, dim=-1), 1)), dim=1)
        return idx


# initializing model

model     = GPT().to(device)
model     = DDP(model, device_ids=[local_rank])
raw_model = model.module

if is_master:
    print(f"params: {sum(p.numel() for p in set(raw_model.parameters()))/1e6:.1f}M")
 
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return min_lr
    coeff = 0.5 * (1.0 + math.cos(math.pi *
            (it - warmup_iters) / (max_iters - warmup_iters)))
    return min_lr + coeff * (learning_rate - min_lr)

# checkpoint
def save_checkpoint(iter_num, best_val_loss):
    path = os.path.join(checkpoint_dir, f"ckpt_{iter_num:07d}.pt")
    torch.save({
        "iter_num":      iter_num,
        "best_val_loss": best_val_loss,
        "model":         raw_model.state_dict(),
        "optimizer":     optimizer.state_dict(),
    }, path)
    print(f"  >> saved {path}")

optimizer = torch.optim.AdamW(raw_model.parameters(),
            lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler()

iter_num, best_val_loss = 0, float("inf")
ckpts = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pt")])
if ckpts:
    ckpt = torch.load(os.path.join(checkpoint_dir, ckpts[-1]),
                      map_location=device, weights_only=False)
    raw_model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    iter_num      = ckpt["iter_num"]
    best_val_loss = ckpt["best_val_loss"]
    if is_master:
        print(f"resumed from iter {iter_num}")

# eval
@torch.no_grad()
def estimate_loss():
    raw_model.eval()
    torch.cuda.empty_cache()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split, bs=eval_batch_size)
            with torch.amp.autocast(device_type="cuda"):
                _, loss = raw_model(x, y)
            losses.append(loss.item())
        out[split] = np.mean(losses)
    torch.cuda.empty_cache()
    raw_model.train()
    return out

# training

if is_master:
    print("\n=== SAMPLE TRAINING BLOCKS ===\n")

    x_sample, _ = get_batch("train", bs=20)

    for i in range(20):
        tokens = x_sample[i].tolist()
        text = enc.decode(tokens)

        print(f"\n--- Sample {i} ---")
        print(text[:300])

if is_master:
    print(f"training from iter {iter_num}...")
t0 = time.time()
optimizer.zero_grad(set_to_none=True)

while iter_num < max_iters:
    for pg in optimizer.param_groups:
        pg["lr"] = get_lr(iter_num)

    if iter_num % eval_interval == 0 and is_master:
        losses = estimate_loss()
        print(f"iter {iter_num:6d} | train {losses['train']:.4f} | "
              f"val {losses['val']:.4f} | lr {get_lr(iter_num):.2e} | "
              f"{(time.time()-t0)/3600:.2f}h")
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            save_checkpoint(iter_num, best_val_loss)
            print(f"  >> best val loss: {best_val_loss:.4f}")

    if iter_num % 5000 == 0 and iter_num > 0 and is_master:
        save_checkpoint(iter_num, best_val_loss)

    for micro_step in range(accumulation_steps):
        x, y = get_batch("train")
        with torch.amp.autocast(device_type="cuda"):
            _, loss = model(x, y)
            loss = loss / accumulation_steps
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    iter_num += 1

if is_master:
    print("done.")
    save_checkpoint(iter_num, best_val_loss)

dist.destroy_process_group()
"""

with open("/kaggle/working/train.py", "w") as f:
    f.write(script)
print("train.py written.")

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/train.py